# Tutorial 13 -- Spin Echo and Dephasing Mitigation

Compare Ramsey and Hahn echo under quasi-static detuning disorder and verify the ensemble-averaged traces against the closed-form theory.

**Prerequisites.** Tutorials 11 and 12 are recommended first.


## 1. Goal

We will model quasi-static detuning disorder explicitly, ensemble-average the resulting pulse-level Ramsey and Hahn-echo traces, and compare them to the textbook closed-form predictions.


## 2. Physical Background

For a fixed detuning offset `delta`, Ramsey gives `P_e^Ramsey(t | delta) = 0.5 (1 + cos(delta t))` while a Hahn echo with a final `-x90` readout pulse refocuses that same static detuning and ideally returns `P_e^Echo(t | delta) = 1`. Averaging over a Gaussian detuning distribution with width `sigma_delta` yields the standard Ramsey envelope `0.5 (1 + exp(-sigma_delta^2 t^2 / 2))`, while the ideal echo stays flat because the static phase cancels exactly.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    cross_kerr_conditional_phase,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    gaussian_quasistatic_echo_excited_population,
    gaussian_quasistatic_ramsey_excited_population,
    ns,
    ramsey_population,
    resonant_drive_excited_population,
    t1_relaxation_population,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
sigma_detuning = 2.0 * np.pi * 0.22e6
detuning_offsets = np.linspace(-4.0, 4.0, 31) * sigma_detuning
detuning_weights = np.exp(-0.5 * (detuning_offsets / sigma_detuning) ** 2)
detuning_weights = detuning_weights / np.sum(detuning_weights)
delays_us = np.linspace(0.0, 8.0, 9)
rotation_duration = 30.0 * ns
rotation_sigma_fraction = 0.18
dt = 4.0 * ns


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=2,
)
x90_pulses, _, _ = build_rotation_pulse(
    RotationGate(index=0, name="x90", theta=np.pi / 2.0, phi=0.0),
    {"duration_rotation_s": rotation_duration, "rotation_sigma_fraction": rotation_sigma_fraction},
)
x180_pulses, _, _ = build_rotation_pulse(
    RotationGate(index=1, name="x180", theta=np.pi, phi=0.0),
    {"duration_rotation_s": rotation_duration, "rotation_sigma_fraction": rotation_sigma_fraction},
)
minus_x90_pulses, _, _ = build_rotation_pulse(
    RotationGate(index=2, name="minus_x90", theta=np.pi / 2.0, phi=np.pi),
    {"duration_rotation_s": rotation_duration, "rotation_sigma_fraction": rotation_sigma_fraction},
)
x90 = x90_pulses[0]
x180 = x180_pulses[0]
minus_x90 = minus_x90_pulses[0]
delays_s = delays_us * us


## 6. Pulse / Sequence Construction


In [ ]:
def simulate_static_detuning_trace(delta_omega):
    frame = FrameSpec(omega_q_frame=model.omega_q + float(delta_omega))
    ramsey_trace = []
    echo_trace = []
    for delay_s in delays_s:
        ramsey_pulses = [
            Pulse(x90.channel, 0.0, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=x90.phase, label="ramsey_a"),
            Pulse(x90.channel, x90.duration + delay_s, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=x90.phase, label="ramsey_b"),
        ]
        ramsey_t_end = 2.0 * x90.duration + delay_s + dt
        ramsey_result = simulate_sequence(
            model,
            SequenceCompiler(dt=dt).compile(ramsey_pulses, t_end=ramsey_t_end),
            model.basis_state(0, 0),
            {"qubit": "qubit"},
            config=SimulationConfig(frame=frame, max_step=dt),
        )
        ramsey_trace.append(final_expectation(ramsey_result, "P_e"))

        echo_pulses = [
            Pulse(x90.channel, 0.0, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=x90.phase, label="echo_a"),
            Pulse(x180.channel, x90.duration + 0.5 * delay_s, x180.duration, x180.envelope, amp=x180.amp, carrier=x180.carrier, phase=x180.phase, label="echo_pi"),
            Pulse(minus_x90.channel, x90.duration + delay_s + x180.duration, minus_x90.duration, minus_x90.envelope, amp=minus_x90.amp, carrier=minus_x90.carrier, phase=minus_x90.phase, label="echo_b"),
        ]
        echo_t_end = 2.0 * x90.duration + x180.duration + delay_s + dt
        echo_result = simulate_sequence(
            model,
            SequenceCompiler(dt=dt).compile(echo_pulses, t_end=echo_t_end),
            model.basis_state(0, 0),
            {"qubit": "qubit"},
            config=SimulationConfig(frame=frame, max_step=dt),
        )
        echo_trace.append(final_expectation(echo_result, "P_e"))

    return np.asarray(ramsey_trace, dtype=float), np.asarray(echo_trace, dtype=float)


## 7. Running the Simulation


In [ ]:
ramsey_mean = np.zeros_like(delays_s, dtype=float)
echo_mean = np.zeros_like(delays_s, dtype=float)
for delta_omega, weight in zip(detuning_offsets, detuning_weights, strict=True):
    ramsey_trace, echo_trace = simulate_static_detuning_trace(delta_omega)
    ramsey_mean += float(weight) * ramsey_trace
    echo_mean += float(weight) * echo_trace

ramsey_theory = gaussian_quasistatic_ramsey_excited_population(delays_s, sigma_detuning)
echo_theory = gaussian_quasistatic_echo_excited_population(delays_s)
print(f"Ramsey ensemble RMS error = {np.sqrt(np.mean((ramsey_mean - ramsey_theory) ** 2)):.3e}")
print(f"Echo ensemble RMS error = {np.sqrt(np.mean((echo_mean - echo_theory) ** 2)):.3e}")


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.plot(delays_us, ramsey_mean, "o-", label="pulse-level Ramsey ensemble")
ax.plot(delays_us, ramsey_theory, "--", color="tab:blue", alpha=0.8, label=r"theory $0.5(1 + e^{-\sigma_\delta^2 t^2 / 2})$")
ax.plot(delays_us, echo_mean, "o-", label="pulse-level Hahn echo ensemble")
ax.plot(delays_us, echo_theory, ":", color="tab:orange", alpha=0.8, label="ideal echo theory")
ax.set_xlabel("Total delay [us]")
ax.set_ylabel(r"Final $P_e$")
ax.set_title("Spin echo refocuses quasi-static detuning")
ax.legend()
plt.show()


## 9. Physical Interpretation

This notebook now isolates the physics that Hahn echo actually refocuses: quasi-static detuning disorder. The middle `pi` pulse cancels the deterministic phase picked up in the first half of the sequence, so the ensemble-averaged echo stays near unity even when Ramsey dephases to `0.5`. Markovian `T_phi` noise is a different process and is not what this tutorial is meant to demonstrate.


## 10. Exercises / Next Steps

- Add a finite `T1` model on top of the static detuning ensemble and watch the echo plateau acquire an irreversible decay envelope.
- Increase `sigma_detuning` and confirm that the Ramsey collapse speeds up while the ideal echo stays nearly unchanged.
- Continue to Tutorial 14 for bosonic Kerr dynamics.
